In [ ]:
import xarray as xr
import dask.dataframe as dd
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data_sparsity.generate_data import GenerateData


def case_paths(case_slug):
    case_root = Path("./tutorial1") / case_slug
    return (
        str(case_root / "netCDF" / "ds.nc"),
        str(case_root / "parquet" / "ddf"),
        str(case_root / "parquet" / "tmp"),
    )


def format_bytes(num_bytes):
    units = ["B", "KiB", "MiB", "GiB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


def parquet_disk_size(parquet_path):
    path = Path(parquet_path)
    if not path.exists():
        path = path.parent
    if path.is_file():
        return path.stat().st_size
    return sum(path_item.stat().st_size for path_item in path.rglob("*") if path_item.is_file())


def plot_grid_case(ds, title):
    x0 = ds["x0"].values
    x1 = ds["x1"].values
    support_x1, support_x0 = np.meshgrid(x1, x0)
    present_mask = np.isfinite(ds["record"].values)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(
        support_x1.ravel(),
        support_x0.ravel(),
        s=160,
        facecolors="none",
        edgecolors="dimgray",
        linewidths=1.2,
        zorder=2,
    )

    for xv in x1:
        ax.axvline(xv, color="dimgray", linestyle=":", linewidth=1, zorder=0)
    for yv in x0:
        ax.axhline(yv, color="dimgray", linestyle=":", linewidth=1, zorder=0)

    ax.scatter(
        support_x1.ravel()[present_mask.ravel()],
        support_x0.ravel()[present_mask.ravel()],
        marker="x",
        c="green",
        s=90,
        linewidths=2,
        zorder=3,
    )

    ax.set_xticks(x1)
    ax.set_yticks(x0)
    ax.set_xticklabels([f"{value:.3f}" for value in x1], color="dimgray")
    ax.set_yticklabels([f"{value:.3f}" for value in x0], color="dimgray")
    ax.set_xlabel("x1", color="dimgray")
    ax.set_ylabel("x0", color="dimgray")
    ax.set_title(title, color="dimgray")
    ax.tick_params(axis="both", colors="dimgray")
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():
        spine.set_color("dimgray")
    ax.grid(False)
    plt.tight_layout()
    plt.show()


def run_case(case_slug, num_obs, sparsity, seed, title):
    ncpath, pqpath, pqpathtmp = case_paths(case_slug)
    gen = GenerateData(
        num_obs=num_obs,
        num_dims=2,
        ratio_dims=1,
        sparsity=sparsity,
        seed=seed,
    )
    gen.generate(
        netcdf_filepath=ncpath,
        parquet_filepath=pqpath,
        parquet_tmp=pqpathtmp,
    )
    ds = xr.open_dataset(ncpath).load()
    df = pd.read_parquet(os.path.dirname(pqpath))

    nc_disk_bytes = Path(ncpath).stat().st_size
    pq_disk_bytes = parquet_disk_size(pqpath)
    ds_memory_bytes = ds.nbytes
    df_memory_bytes = df.memory_usage(index=True, deep=True).sum()

    print(f"Loaded netCDF into xarray: {format_bytes(ds_memory_bytes)} in memory")
    print(f"Loaded parquet into pandas: {format_bytes(df_memory_bytes)} in memory")
    print(f"On-disk netCDF size: {format_bytes(nc_disk_bytes)}")
    print(f"On-disk parquet size: {format_bytes(pq_disk_bytes)}")
    plot_grid_case(ds, title)
    return ds, df

### List of cases - Single variable

2D because it's easier to understand and visualize

* Purely gridded data, maximum density (minimum sparsity)
* Purely irregular data
* Maximum sparsity on a grid
* Somehow sparse data on a grid

Each case is written to its own folder and rendered as a representative map.

### Purely gridded data, 3x3 grid, 9 points

In [ ]:
ds = run_case(
    "purely_gridded",
    num_obs=9,
    sparsity=1,
    seed=202607,
    title="Purely gridded data",
)

### Purely irregular data, 3 points

In [ ]:
ds = run_case(
    "purely_irregular",
    num_obs=3,
    sparsity=1 / 3,
    seed=202607,
    title="Purely irregular data",
)


### Maximum sparsity on a grid, 3 points

In [ ]:
ds = run_case(
    "maximum_sparsity_grid",
    num_obs=3,
    sparsity=1 / 3,
    seed=202607,
    title="Maximum sparsity on a grid",
)


### Somehow sparse data on a grid, 6 points

In [ ]:
ds = run_case(
    "somehow_sparse_grid",
    num_obs=6,
    sparsity=6 / 9,
    seed=202607,
    title="Somehow sparse data on a grid",
)
